# 1. Contextualização

O desmatamento na Amazônia Legal representa uma importante questão ambiental, devido aos seus impactos sobre a biodiversidade, o equilíbrio climático e a preservação dos ecossistemas. O monitoramento da evolução das áreas desmatadas ao longo do tempo é fundamental para compreender a dinâmica desse fenômeno e identificar padrões relevantes.

Neste contexto, o presente trabalho tem como objetivo realizar uma análise exploratória dos dados de desmatamento disponibilizados pelo Projeto de Monitoramento do Desmatamento na Amazônia Legal por Satélite (PRODES), considerando a evolução temporal e a distribuição entre os estados que compõem a região. Por meio da utilização de Python e de bibliotecas voltadas à análise e visualização de dados, busca-se identificar tendências e padrões a partir das informações disponíveis no conjunto de dados.

A análise será orientada pelas seguintes perguntas de pesquisa:

RQ1: Como o desmatamento na Amazônia Legal evoluiu ao longo do período analisado?

RQ2: Quais estados da Amazônia Legal apresentam os maiores índices de desmatamento acumulado no período analisado?

A partir dessas questões, pretende-se produzir informações que contribuam para uma melhor compreensão da distribuição e da evolução histórica do desmatamento na região.

# 2. Importação e leitura dos dados

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/raw/desmatamento_prodes.csv")

df.head()

,referencia,acre,amazonas,amapa,maranhao,mato_grosso,para,rondonia,roraima,tocantins,area_total_desmatamento
0,1988,620,1510,60,2450,5140,6990,2340,290,1650,21050
1,1989,540,1180,130,1420,5960,5750,1430,630,730,17770
2,1990,550,520,250,1100,4020,4890,1670,150,580,13730
3,1991,380,980,410,670,2840,3780,1110,420,440,11030
4,1992,400,799,36,1135,4674,3787,2265,281,409,13786


# 3. Verificação inicial da base de dados

In [65]:
# entendendo melhor sobre a base de dados, verificando se existem valores ausentes e duplicados

print("Dimensões da base: ", df.shape)
print("\nTipos de dados: ")
print(df.dtypes)

print("\nValores ausentes: ")
print(df.isnull().sum())

print("\nDuplicidades: ")
print(df.duplicated().sum())

colunas = df.columns.drop("referencia")

(df[colunas] < 0).sum()

Dimensões da base:  (35, 11)

Tipos de dados: 
referencia                 int64
acre                       int64
amazonas                   int64
amapa                      int64
maranhao                   int64
mato_grosso                int64
para                       int64
rondonia                   int64
roraima                    int64
tocantins                  int64
area_total_desmatamento    int64
dtype: object

Valores ausentes: 
referencia                 0
acre                       0
amazonas                   0
amapa                      0
maranhao                   0
mato_grosso                0
para                       0
rondonia                   0
roraima                    0
tocantins                  0
area_total_desmatamento    0
dtype: int64

Duplicidades: 
0


acre                       0
amazonas                   0
amapa                      0
maranhao                   0
mato_grosso                0
para                       0
rondonia                   0
roraima                    0
tocantins                  0
area_total_desmatamento    0
dtype: int64

In [66]:
colunas_estados = [
    "acre",
    "amazonas",
    "amapa",
    "maranhao",
    "mato_grosso",
    "para",
    "rondonia",
    "roraima",
    "tocantins"
]

# verificando se as somas dos valores realmente estao corretos

df["soma_estados"] = df[colunas_estados].sum(axis=1)

df["diferenca"] = (
    df["area_total_desmatamento"] - df["soma_estados"]
)

df[df["diferenca"] != 0]

,referencia,acre,amazonas,amapa,maranhao,mato_grosso,para,rondonia,roraima,tocantins,area_total_desmatamento,soma_estados,diferenca


In [67]:
df = df.drop(columns=["soma_estados", "diferenca"])

In [68]:
# trocando o nome das colunas de referencia e area_total_desmatamento

df = df.rename(columns={
    "referencia" : "ano",
    "area_total_desmatamento" : "area_total_desmatada"
})

# 4. Estatísticas descritivas

In [69]:
# entendendo melhor as principais estatisticas sobre os dados apresentados

df.describe()

,ano,acre,amazonas,amapa,maranhao,mato_grosso,para,rondonia,roraima,tocantins,area_total_desmatada
count,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000,35.000000
mean,2005.000000,500.428571,954.200000,47.485714,753.857143,4344.485714,4764.371429,1889.571429,261.400000,251.142857,13766.942857
std,10.246951,251.248329,547.907667,78.469691,502.825086,3074.753719,1771.290873,1000.094563,131.758514,319.975748,6394.963542
min,1988.000000,167.000000,370.000000,0.000000,209.000000,757.000000,1741.000000,435.000000,84.000000,23.000000,4571.000000
25%,1996.500000,307.000000,592.000000,13.000000,343.000000,1581.000000,3775.000000,1189.500000,177.000000,51.000000,7714.500000
50%,2005.000000,441.000000,775.000000,25.000000,674.000000,4020.000000,4890.000000,1670.000000,231.000000,156.000000,13227.000000
75%,2013.500000,606.000000,1154.500000,42.500000,1063.000000,6417.500000,5789.500000,2448.500000,303.000000,326.500000,17965.500000
max,2022.000000,1208.000000,2607.000000,410.000000,2450.000000,11814.000000,8870.000000,4730.000000,630.000000,1650.000000,29059.000000


In [70]:
estados = [
    "acre", "amazonas", "amapa", "maranhao",
    "mato_grosso", "para", "rondonia",
    "roraima", "tocantins"
]

df_acumulado = pd.DataFrame({
    "estado" : estados,
    "desmatamento_acumulado" : df[estados].sum().values
    })

In [71]:
df_acumulado = df_acumulado.sort_values(by="desmatamento_acumulado", ascending=False)

df_acumulado

,estado,desmatamento_acumulado
5,para,166753
4,mato_grosso,152057
6,rondonia,66135
1,amazonas,33397
3,maranhao,26385
0,acre,17515
7,roraima,9149
8,tocantins,8790
2,amapa,1662


In [72]:
df_estatisticas_estados = df[estados].describe().T

df_estatisticas_estados = df_estatisticas_estados[
    ["mean", "50%", "std", "min", "max"]
]

display(df_estatisticas_estados)

,mean,50%,std,min,max
acre,500.428571,441.0,251.248329,167.0,1208.0
amazonas,954.200000,775.0,547.907667,370.0,2607.0
amapa,47.485714,25.0,78.469691,0.0,410.0
maranhao,753.857143,674.0,502.825086,209.0,2450.0
mato_grosso,4344.485714,4020.0,3074.753719,757.0,11814.0
para,4764.371429,4890.0,1771.290873,1741.0,8870.0
rondonia,1889.571429,1670.0,1000.094563,435.0,4730.0
roraima,261.400000,231.0,131.758514,84.0,630.0
tocantins,251.142857,156.0,319.975748,23.0,1650.0


In [73]:
df_participacao_estados = df[estados].div(
    df["area_total_desmatada"], axis=0
) * 100

df_participacao_estados.insert(
    0, "ano", df["ano"]
)

df_participacao_estados.style.format({
    estado: "{:.2f}%" for estado in estados
})

,ano,acre,amazonas,amapa,maranhao,mato_grosso,para,rondonia,roraima,tocantins
0,1988,2.95%,7.17%,0.29%,11.64%,24.42%,33.21%,11.12%,1.38%,7.84%
1,1989,3.04%,6.64%,0.73%,7.99%,33.54%,32.36%,8.05%,3.55%,4.11%
2,1990,4.01%,3.79%,1.82%,8.01%,29.28%,35.62%,12.16%,1.09%,4.22%
3,1991,3.45%,8.88%,3.72%,6.07%,25.75%,34.27%,10.06%,3.81%,3.99%
4,1992,2.90%,5.80%,0.26%,8.23%,33.90%,27.47%,16.43%,2.04%,2.97%
5,1993,3.24%,2.48%,0.00%,2.50%,41.76%,28.76%,17.42%,1.61%,2.24%
6,1994,3.24%,2.48%,0.00%,2.50%,41.76%,28.76%,17.42%,1.61%,2.24%
7,1995,4.16%,7.27%,0.03%,6.01%,35.76%,27.00%,16.28%,0.76%,2.74%
8,1996,2.38%,5.63%,0.00%,5.84%,36.03%,33.78%,13.39%,1.18%,1.76%
9,1997,2.71%,4.45%,0.14%,3.09%,39.85%,31.29%,15.01%,1.39%,2.06%


# 5. Análise Exploratória

In [77]:
# carregando o arquivo de dados tratados

df = pd.read_csv("../data/processed/desmatamento_tratado.csv")

### Evolução do desmatamento ao longo do ano

In [78]:
fig = px.line(
    df,
    x="ano",
    y="area_total_desmatada",
    title="Evolução do desmatamento na Amazônia Legal",
    labels={
        "ano": "Ano",
        "area_total_desmatada": "Área desmatada (km²)"
    },
    markers=True
)

fig.show()

### Desmatamento acumulado por estado

In [79]:
fig = px.bar(
    df_acumulado,
    x="desmatamento_acumulado",
    y="estado",
    orientation="h",
    title="Desmatamento acumulado por estado (1988–2022)",
    labels={
        "estado": "Estado",
        "desmatamento_acumulado": "Área desmatada acumulada (km²)"
    }
)

fig.update_layout(
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

### Evolução do desmatamento por estado

In [80]:
df_estados_ano = df.melt(
    id_vars="ano",
    value_vars=estados,
    var_name="estado",
    value_name="area_desmatada"
)

fig = px.line(
    df_estados_ano,
    x="ano",
    y="area_desmatada",
    color="estado",
    title="Evolução do desmatamento por estado",
    labels={
        "ano": "Ano",
        "area_desmatada": "Área desmatada (km²)",
        "estado": "Estado"
    }
)

fig.show()

### Interpretação dos gráficos

A análise da evolução temporal do desmatamento na Amazônia Legal evidencia uma tendência de redução da área desmatada a partir do final da década de 2000, especialmente quando comparada aos níveis observados durante as décadas de 1980 e 1990. Essa mudança indica uma alteração significativa no comportamento do desmatamento ao longo da série histórica, embora ainda seja possível observar oscilações entre determinados anos.

Em relação à distribuição espacial do desmatamento, os estados do Pará e de Mato Grosso se destacam como os principais responsáveis pela área desmatada acumulada no período analisado. Ambos apresentam valores consideravelmente superiores aos demais estados, evidenciando uma concentração do desmatamento nesses territórios.